# Campaign group analysis

Load the saved Step 3 graph, IO labels, classifier predictions, and node indicators for one campaign. Then build and aggregate the five graph-based account groups.

In [ ]:
from pathlib import Path

import networkx as nx
import pandas as pd

## Parameters

In [ ]:
campaign_name = "Iran_1"

## Locate the saved Step 3 files

This analysis uses the standard Step 3 output with `BERTopic_topic_256` and a top percentage of 100.

In [ ]:
def find_project_root(start_path: str | Path) -> Path:
    """Find the project root containing the src and results folders.

    Args:
        start_path: Directory from which to search upward.

    Returns:
        Project root directory.

    Raises:
        FileNotFoundError: If the project root cannot be found.
    """
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")


project_root = find_project_root(Path.cwd())
topic_col = "BERTopic_topic_256"
top_percentage = 100
dataset_part = f"{campaign_name}_part_all"
step_3_path = (
    project_root
    / "results"
    / "Labeled_Datasets"
    / campaign_name
    / dataset_part
    / "step_3_key_authors_graph"
    / f"topic_col_{topic_col}_top_percentage_{top_percentage:g}"
)
graph_stem = f"accounts_graph_topic_col_{topic_col}"
graph_path = step_3_path / f"{graph_stem}.graphml"
node_indicators_path = step_3_path / f"{graph_stem}_node_indicators.parquet"
data_path = step_3_path / f"{dataset_part}_step_3.gzip.parquet"
classifier_predictions_path = (
    step_3_path
    / "io_classification_random_forest_all_indicators_cv_predictions.csv"
)

input_paths = {
    "graph": graph_path,
    "node indicators": node_indicators_path,
    "post data": data_path,
    "classifier predictions": classifier_predictions_path,
}
missing_paths = [path for path in input_paths.values() if not path.is_file()]
if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Missing Step 3 input files:\n{missing_text}")

pd.DataFrame.from_dict(input_paths, orient="index", columns=["path"])

In [ ]:
# analysis_output_path = project_root / "result" / "group_analysis" / campaign_name
group_analysis_folder_path = step_3_path / "group_analysis"

## Import the group-analysis functions

Definition-only mode imports the functions without running the Step 3 demonstrations or rebuilding data.

In [ ]:
starting_directory = Path.cwd()
%cd {project_root / "src"}
key_authors_graph_definition_only = True
%run -i 3_key_authors_graph.ipynb
%cd {starting_directory}

## Load the graph, labels, predictions, and indicators

In [ ]:
G_accounts = nx.read_graphml(graph_path)
node_indicators_df = pd.read_parquet(node_indicators_path)
node_indicators_df.index = node_indicators_df.index.astype(str)
df = pd.read_parquet(
    data_path,
    columns=["accountid", "post_time", topic_col, f"{topic_col}_top_words"],
)

classifier_predictions_df = pd.read_csv(
    classifier_predictions_path,
    dtype={"author_id": str},
)
io_authors_set = set(
    classifier_predictions_df.loc[
        classifier_predictions_df["is_io"].eq(1),
        "author_id",
    ]
)
classifier_io_authors_set = set(
    classifier_predictions_df.loc[
        classifier_predictions_df["predicted_is_io"].eq(1),
        "author_id",
    ]
)

load_summary_df = pd.DataFrame(
    {
        "value": [
            G_accounts.number_of_nodes(),
            G_accounts.number_of_edges(),
            len(node_indicators_df),
            len(io_authors_set),
            len(classifier_io_authors_set),
        ]
    },
    index=[
        "graph nodes",
        "graph edges",
        "node indicator rows",
        "ground-truth IO authors",
        "classifier-predicted IO authors",
    ],
)
load_summary_df.style.format({"value": "{:,}"})

## Group analysis

### All Authors

In [ ]:
all_node_group_analysis_df = build_group_analysis(
    nodes=set(G_accounts.nodes),
    G=G_accounts,
    io_authors_set=io_authors_set,
    classifier_io_authors_set=classifier_io_authors_set,
    node_indicators_df=node_indicators_df,
    output_path = group_analysis_folder_path,
    output_filename="all_nodes_group_analysis_results.csv",
)

group_analysis_aggregate_df = aggregate_and_plot_group_analysis(
    group_analysis_df=all_node_group_analysis_df,
    output_path=group_analysis_folder_path,
    file_name_prefix="all_nodes",
)

### Io Authors

In [ ]:
rf_io_authors_group_analysis_df = build_group_analysis(
    nodes=set(classifier_io_authors_set),
    G=G_accounts,
    io_authors_set=io_authors_set,
    classifier_io_authors_set=classifier_io_authors_set,
    node_indicators_df=node_indicators_df,
    output_path = group_analysis_folder_path,
    output_filename="rf_io_authors_group_analysis_results.csv",
)

rf_io_authors_group_analysis_aggregate_df = aggregate_and_plot_group_analysis(
    group_analysis_df=rf_io_authors_group_analysis_df,
    output_path=group_analysis_folder_path,
    file_name_prefix="rf_io_authors",
)

## Analyze Group Topics

In [ ]:
top_core_authors_df = get_top_authors_by_indicator(
    node_indicators_df,
    indicator_col="core_number",
    n=10,
)
top_core_user = top_core_authors_df.index[0]

core_numbers = nx.core_number(G_accounts)
top_core_user_k_core_nodes = get_node_k_core(
    node_id=top_core_user,
    G=G_accounts,
    core_numbers=core_numbers,
    include_node=False,
)

print(top_core_user_k_core_nodes)

In [ ]:
group_topic_counts_df = analyze_group_topics(
    df=df,
    nodes=top_core_user_k_core_nodes,
    time_window="3D",
    topic_col=topic_col,
    output_path=group_analysis_folder_path / "top_core_user_k_core_group_topics",
)